# Tilt Correction

Almost every step of a reduction assumes that a **column of pixels is a single wavelength**.
Chapter 7 will subtract sky measured 60 rows from the source as if it were at the source's
wavelengths, and Chapter 10 will apply a wavelength solution fitted in one row to a spectrum
extracted from another. [Chapter 5](05-arc-lamps.ipynb) showed that on these frames the
assumption fails: the arc lines lean by several pixels across the slit, a tilt distortion set by
the spectrograph's optics. This chapter measures the distortion, models it, and removes it, before
anything else is measured from the frames.

`specreduce.tilt_correction.TiltCorrection` finds arc lines at a set of rows along the slit, fits
a 2D polynomial mapping a rectified coordinate space to the detector, and (optionally) resamples each frame
onto the rectified grid, where columns are wavelengths again. The product is a `TiltSolution`,
saved to disk for every later chapter. Getting this step wrong has consequences. As Chapter 1
recounts, an insufficient tilt correction in the original reduction of these data left a
Rayleigh-like slope in the transmission spectrum published by [Parviainen et al.
(2016)](https://ui.adsabs.harvard.edu/abs/2016A%26A...585A.114P/abstract), the kind of
wavelength-dependent systematic that can mask the true signal from a planetary atmosphere.

:::{note}
**This step is optional.** Whether you need it depends on the instrument. The OSIRIS tilt is
unusually strong, which makes these frames a good teaching case but not a typical one, and the
section "Do you need this step at all?" shows how to decide for your own data. Resampling is a
choice too: the fitted `TiltSolution` is a coordinate transformation, exported as a GWCS and to
ASDF, that a reduction working on the unrectified frame can use without resampling at all. This
book rectifies, because the background subtraction and extraction that follow assume a column is
a single wavelength; Chapter 12 shows the other route.
:::

In [ ]:
import sys

sys.path.append("../src")

import numpy as np
import matplotlib.pyplot as plt
from astropy.visualization import simple_norm
from specreduce.tilt_correction import TiltCorrection
from specreduce.tracing import FlatTrace

from common import TILT_SOLUTION_FILE, read_data

plt.rc("figure", figsize=(8, 2.4), dpi=110)
plt.rc("font", size=9)

arcs, lamps, obj = read_data()
ny, nx = obj.data.shape

## Seeing the tilt

Arc lamps make the distortion obvious, because an arc frame is nothing *but* emission lines. Each
line is a monochromatic image of the slit, so it should be a perfectly vertical bar. Here is the
Ne frame, stretched so the lines are easy to follow.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.2), constrained_layout=True)
ne = arcs[lamps.index("Ne")].data

ax.imshow(
    ne,
    origin="lower",
    aspect="auto",
    cmap=plt.cm.Blues,
    interpolation="bicubic",
    norm=simple_norm(ne, stretch="log", vmin=0, vmax=150_000),
)
ax.set(
    xlim=(690, 800),
    ylim=(0, 371),
    xlabel="Dispersion [pix]",
    ylabel="Cross-dispersion [pix]",
)
# Ticks on the top spine as well, so the lean can be read off at both ends of the slit.
ax.tick_params(axis="x", top=True, labeltop=True);

The lines lean visibly. To put a number on it, we take the intensity-weighted centroid of one
bright Ne line in 20-row bands along the slit: it moves by 5.6 pixels between the ends of the
illuminated slit, more than twice the 2.5-pixel line width.

## Setting up `TiltCorrection`

`TiltCorrection` needs the arc frames and a **reference row**, which defines the rectified space:
after correction every row has the wavelength scale this row had originally. It should sit where
the science spectrum falls, so that the rows you extract are the ones the rectification moves
least. We use row 135, where Chapter 4 found the source, expressed as a `FlatTrace`. It is a
reference row, not a trace, so it does not follow the target's curvature.

In [ ]:
reference = FlatTrace(arcs[0], 135)

tilt = TiltCorrection(arcs, reference, n_cdisp_samples=7)

print(f"reference pixel (cross-disp, disp): {tilt.ref_pixel}")
print(f"cross-dispersion samples:           {tilt.cd_samples}")
print(f"arc frames:                         {tilt.nframes}")

Two arguments shape the sampling. `n_cdisp_samples` sets how many rows are sampled across the
slit; the seven chosen rows are listed above. `cdisp_sample_lims` bounds the region they are drawn
from, and is left at its default, the whole frame, because the loader has already trimmed the
frame to `TRIMSEC` (Chapter 2). On an untrimmed frame it is how you keep the samples out of the
prescan and the unilluminated strip below the slit, where there are no arc lines to find.

We pass all three lamps together. The tilt is a property of the **optics**, not of any
particular lamp, so every arc frame measures the same distortion; using all three
provides more lines and better coverage along the dispersion axis.

## Finding the arc lines

`find_arc_lines` detects emission peaks at the reference row and at each sampled row with the
threshold-based line finder of `specutils`. Two parameters control it: `fwhm`, the expected line
width, here the 2.5 pixels measured in Chapter 5, and `noise_factor`, how far above the noise a
peak must rise to count. The threshold applies to the flux itself, and an arc row does not sit at
zero: scattered light puts a pedestal of 100 to 180 e⁻ under the lines, 6 to 10 times the
per-pixel uncertainty, enough for noise on the baseline to register as lines. `find_arc_lines`
therefore subtracts a sigma-clipped median baseline from every row before thresholding. That is
the default (`subtract_baseline=True`); `baseline_window` makes the baseline a running estimate,
and `subtract_baseline=False` is for arcs that were background-subtracted before they got here.

We use `noise_factor=25`, a generous margin. These arcs are bright enough that it costs only the
faintest detections, and every remaining detection is a well-measured line.


In [ ]:
tilt.find_arc_lines(fwhm=2.5, noise_factor=25)

fig, axs = plt.subplots(3, figsize=(8, 5.4), sharex="all", constrained_layout=True)

for ax, frame, lamp, detector_x, detector_y, reference_lines in zip(
    axs, arcs, lamps, tilt._samples_det_x, tilt._samples_det_y, tilt._lines_ref
):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    ax.plot(detector_x, detector_y, "k.", ms=2, label="detector-space samples")
    ax.plot(
        reference_lines,
        np.full_like(reference_lines, tilt.ref_pixel[0]),
        "rx",
        ms=5,
        label="reference row",
    )
    ax.text(0.01, 0.9, lamp, transform=ax.transAxes, va="top", size=8)
    ax.set_ylabel("Cross-disp. [pix]")

axs[-1].legend(frameon=False, loc="lower left", fontsize=7, ncols=2)
axs[-1].set_xlabel("Dispersion [pix]");

Each panel carries two sets of points. The red **reference lines** are those found in row 135,
and they define the rectified coordinates. The black **detector-space samples** are every line
found at every sampled row, and they show where those wavelengths actually land: each
near-vertical chain of dots is one line, and the chains lean consistently, tracing the tilt
directly. A faint line may be found at some rows and missed at others, so the fit matches the two
sets with a kd-tree rather than assuming they correspond one to one. The lamps populate different
parts of the spectrum, HgAr toward the blue and Xe toward the red, so together they cover the
whole dispersion axis.

## Fitting the transformation

`fit` finds the 2D polynomial mapping rectified coordinates to detector coordinates in two stages:
a robust kd-tree-based minimization to get close, then a least-squares fit to matched line pairs.
`refine_fit` repeats the second stage with a tighter `match_distance_bound`, because the first
pass has to be generous enough to find matches at all.

In [ ]:
tilt.fit(degree=4)
tilt.refine_fit(degree=4, match_distance_bound=1.0)

`match_distance_bound` is the one setting here that deserves thought, because it decides which
pairs the refit sees. It has to sit between two scales: the centroiding scatter of the lines,
about 0.04 pixels, and the spacing of neighboring arc lines, which gets down to 3 to 5 pixels in
the dense parts of the Ne and Xe spectra. A bound of several pixels pairs lines with their
neighbors. At 5 pixels about 20 such mismatches enter the fit, the rms of the matched residuals
rises from 0.04 to 0.55 pixels, and the solution shifts by up to 2.7 pixels across the frame.
Anything from a quarter of a pixel to 2 pixels gives solutions that agree to better than a quarter
of a pixel everywhere on the detector. We use **1.0 pixel**, well above the centroiding scatter
and well below the line spacing, and the model then reproduces the matched positions to about
**0.04 pixels**.

`plot_fit_quality` shows the matched lines on the detector, with marker size proportional to the
residual, and the residuals against each axis in the margins. It matches at its own default bound
of 5 pixels, so the few large markers are the mismatched pairs just described.

In [ ]:
tilt.plot_fit_quality(figsize=(8, 5), rlim=(-0.5, 0.5))

`plot_wavelength_contours` draws the transformation in detector space as curves of constant
rectified wavelength, the lines that *should* have been vertical.

In [ ]:
fig, axs = plt.subplots(2, figsize=(8, 4.2), sharex="all", constrained_layout=True)

for ax, frame, label in ((axs[0], arcs[1], "Ne arc"), (axs[1], obj, "science")):
    ax.imshow(
        frame.data,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        norm=simple_norm(frame.data, stretch="log", vmin=0, vmax=250_000),
    )
    tilt.plot_wavelength_contours(ax=ax)
    ax.text(0.01, 0.92, label, transform=ax.transAxes, va="top", size=8)
    ax.set(xlim=(0, nx - 1), ylabel="Cross-disp. [pix]")

axs[-1].set_xlabel("Dispersion [pix]");

The contours bow away from vertical by several pixels at the ends of the slit and lie on the arc
lines in the upper panel, which confirms that the model has captured the real distortion. On the
science frame they show the distortion that was silently present when Chapter 4 traced it.

## Rectifying the frames

`solution.resample` applies the transformation, rebinning each frame onto the rectified grid with
an exact, flux-conserving algorithm. We rectify the three arcs and the science frame, and compare
the total flux before and after on the Ne arc and the science frame.

In [ ]:
rectified = {
    label: tilt.solution.resample(frame)
    for label, frame in zip(lamps + ("science",), list(arcs) + [obj])
}

for label, frame in (("Ne", arcs[1]), ("science", obj)):
    before = frame.data.sum()
    after = rectified[label].data.sum()
    print(
        f"{label:>8}: flux before {before:.6e}, after {after:.6e}, "
        f"ratio {after / before:.6f}"
    )


Flux is conserved to better than one part in two thousand; the small loss is light that falls
outside the rectified grid at the frame edges. `resample` carries the whole `NDData` across, not
only the flux: the propagated variance, a mask marking any output bin that could not be filled,
and the metadata, including the read noise. The uncertainty matters immediately, because
`Background` (Chapter 7) needs one to propagate and `HorneExtract` (Chapter 8) refuses to run
without one.

The Ne frame before and after rectification:

In [ ]:
fig, axs = plt.subplots(
    2, figsize=(8, 4.4), sharex="all", sharey="all", constrained_layout=True
)

for ax, image, label in (
    (axs[0], arcs[1].data, "Ne, raw"),
    (axs[1], rectified["Ne"].data, "Ne, rectified"),
):
    ax.imshow(
        image,
        origin="lower",
        aspect="auto",
        cmap=plt.cm.Blues,
        interpolation="bicubic",
        norm=simple_norm(image, stretch="log", vmin=0, vmax=150_000),
    )
    ax.text(
        0.01,
        0.93,
        label,
        transform=ax.transAxes,
        va="top",
        size=8,
        bbox=dict(fc="white", ec="none", alpha=0.7, pad=1.5),
    )
    ax.set_ylabel("Cross-disp. [pix]")

axs[-1].set(xlim=(690, 800), ylim=(0, 371), xlabel="Dispersion [pix]");

Tracked through the rectified frame, the bright Ne line from "Seeing the tilt" now drifts by well
under 0.1 pixels along the slit, down from 5.6. The same wavelength lands in the same column in
every row, which is the property every later step assumes.

:::{warning}
**A resampled variance is not the variance of an independent measurement.** Rectification splits
each detector pixel between the output bins it straddles, and a bin that receives a fraction $f$
of a pixel holding $N$ electrons receives $fN$ of the flux but only $f^2N$ of the variance. The
propagated variance is therefore *sub-Poisson*, about 40 percent below the independent-pixel
model on these frames. That is the correct diagonal of the covariance, but no per-pixel variance
can express the off-diagonal: neighboring columns of a rectified frame share noise, so noise
estimators based on pixel-to-pixel differences read low. Chapter 9 meets a consequence of the
sub-Poisson diagonal, a detection threshold in units of the uncertainty that is lower after
rectification than before. Estimate your noise before resampling where you can, and treat
per-pixel errors on a rectified frame as a lower bound.
:::

## Choosing the parameters

`degree` is the order of the 2D polynomial. Optical distortion is smooth, so low orders suffice:
`fit` accepts no less than 3, because it seeds its refinement from an internal degree-3 model, and
higher orders buy nothing and can oscillate between the sampled rows. We use 4, which leaves a
little headroom.

`n_cdisp_samples` must stay comfortably above `degree`, since a polynomial of degree $d$ in the
cross-dispersion coordinate needs more than $d + 1$ sample rows to be determined. With fewer, the
fit passes exactly through its constraints, reports a tiny residual, and typically maps the
unsampled rows far off the detector, so that `resample` finds nothing to put there. **A low
residual on an underdetermined fit is a symptom, not a reassurance.** Judge the fit on the
rectified frame instead: the flux should still be there, and a given line should land in the same
column in every row. Seven samples across the slit do that here; more cost line-finding time and
change nothing.

`arc_frames` accepts one frame or several. One lamp is enough in principle, since every lamp
measures the same distortion, and more lamps add lines over more of the dispersion range. Do not
compare the options by fit residual: it measures the centroiding scatter of whichever lines a lamp
offers, so a lamp with a few bright, isolated lines can post a lower rms than three lamps together
while constraining the transformation less well.

## Do you need this step at all?

The tilt on these frames is severe, but that is a property of this spectrograph, not of long-slit
spectroscopy. A well-aligned instrument can show a fraction of a pixel of drift across the whole
slit, and then rectifying gains nothing and costs a resampling. The test is cheap: take the
centroid of one arc line in successive bands of rows along the slit, as in "Seeing the tilt", and
divide the total drift by the width of the line. Here that is 5.6 pixels against 2.5, a ratio of
about **2.2**. As a rule of thumb:

| drift / line width | what it means |
|---|---|
| well below 0.5 | The distortion is smaller than the centroiding error; skip the correction. |
| around 0.5 to 1 | Tolerable for wavelength calibration, but it shows up in sky subtraction if the science slit is narrow enough to give sharp sky lines. |
| above 1 | Correct it. A wavelength lands in different columns along the slit, and nothing downstream can recover from that. |

These frames are well into the third case. The ratio decides, not either number alone: the drift
depends on the optical design and the line width on the slit, so a narrower slit makes the same
distortion matter more.

:::{note}
**If you skip this step, nothing downstream changes except the numbers.** The chapters that follow
call `Background`, `BoxcarExtract`, `HorneExtract` and `WavelengthCalibration1D` on the rectified
frames, but none of them depends on the rectification. Substituting the raw `obj` and `arcs` works
as well, and is the right choice if your instrument does not need the correction.
:::

## Saving the solution

A tilt solution is a calibration product. It depends on the arc frames and the instrument's
optics, not on any science exposure, so it is fitted once and applied to every frame taken in the
same configuration, which in a real program can mean hundreds of exposures. Refitting it for each
would be wasteful and would let the solution drift between frames that ought to share it.

`TiltSolution.to_asdf` writes the solution to [ASDF](https://asdf-standard.readthedocs.io/), the
format Astropy uses for structured scientific data, and `TiltSolution.from_asdf` reads it back. The
file holds the transformation as an **analytic model**, a few kilobytes of polynomial coefficients
rather than a resampled image, so the restored solution is the same polynomial and resamples a
frame to the same flux and variance.

In [ ]:
tilt.solution.to_asdf(TILT_SOLUTION_FILE)

Every chapter after this one starts by reading it back:

In [ ]:
from specreduce.tilt_solution import TiltSolution
from common import TILT_SOLUTION_FILE

solution = TiltSolution.from_asdf(TILT_SOLUTION_FILE)

The model is stored as a [GWCS](https://gwcs.readthedocs.io/) object, the same one available live
as `TiltSolution.gwcs`. Its forward transform maps rectified `(disp, cdisp)` pixel coordinates to
detector coordinates, with the cross-dispersion coordinate unchanged, and the inverse is attached
when the image shape is known. Any GWCS-aware code can therefore evaluate the transformation
without importing `specreduce`, for example to extract on the unrectified frame. The file and the
live property differ in one respect: the file carries the image shape as the GWCS bounding box,
while `gwcs` is left unbounded, because a bounded GWCS returns `NaN` outside its box instead of
extrapolating and would blank the frame edges.

## Summary

- The **tilt distortion** makes wavelength vary along the slit. On these frames an arc line drifts
  **5.6 pixels** between the ends of the illuminated slit, more than twice its 2.5-pixel width,
  and under 0.1 pixels after rectification.
- `TiltCorrection` finds arc lines at several rows, fits a 2D polynomial from rectified to
  detector space, and `resample` rebins each frame onto the rectified grid, conserving flux to
  better than 0.1 percent.
- The rectified frame keeps its variance, mask and metadata. The variance is sub-Poisson, and it
  cannot express the correlation resampling introduces between neighboring columns.
- Set the reference row where the science spectrum falls: it defines the wavelength scale of the
  rectified frame, and the rows near it are the ones rectification moves least.
- `degree=4`, `n_cdisp_samples=7` and a 1-pixel `match_distance_bound` work here. Keep the
  samples comfortably above the degree and the bound well inside the line spacing, and judge the
  fit on the rectified frame rather than on its residual. One lamp is enough; more buy coverage.
- **Save the solution, do not refit it.** `to_asdf` writes it as an analytic model, and
  `TiltSolution.gwcs` exposes the same transformation to any GWCS-aware tool.
- **The step is optional.** Divide the drift of an arc line along the slit by its width; if the
  ratio is well below 0.5, skip the correction.

Next: [Background Subtraction](07-background.ipynb), which measures the sky in rows either side of
the trace, now that each column is a single wavelength.